# Weekly Project 5 
## Implementation of global registration 
### Task 1
Today, your task is to implement a global registration algorithm.

It should be able to roughly align two point clouds.
Implement the global registration, and then try the following:

1. Can you fit `r1.pcd` and `r2.pcd`?
2. Can you fit `car1.ply` and `car2.ply`?
The corresponding files are in the `global_registration` folder.


### Task 2 (Challange)
Challanges attempt either or both:
- Implement local registration.

- Attempt to reconstruct the car from the images in `car_challange` folder.

You can use the notebooks from Monday as a starting point.

In [5]:
import open3d as o3d
import numpy as np
import copy
import os

# Counter for generating unique filenames
_vis_counter = 0

# Helper function for saving point cloud visualizations
def draw_registrations(source, target, transformation=None, recolor=False, filename=None):
    """
    Save visualization of two point clouds with optional transformation.

    Args:
        source: Source point cloud
        target: Target point cloud
        transformation: Optional 4x4 transformation matrix to apply to source
        recolor: If True, color source orange and target blue for clarity
        filename: Optional filename to save image (if None, auto-generates name)
    """
    global _vis_counter

    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)

    if recolor:
        source_temp.paint_uniform_color([1, 0.706, 0])  # Orange
        target_temp.paint_uniform_color([0, 0.651, 0.929])  # Blue

    if transformation is not None:
        source_temp.transform(transformation)

    # Create output directory if it doesn't exist
    output_dir = "visualizations"
    os.makedirs(output_dir, exist_ok=True)

    # Generate filename if not provided
    if filename is None:
        filename = f"registration_{_vis_counter:03d}.png"
        _vis_counter += 1

    filepath = os.path.join(output_dir, filename)

    # Render and save using offscreen rendering
    vis = o3d.visualization.Visualizer()
    vis.create_window(visible=False)  # Create hidden window
    vis.add_geometry(source_temp)
    vis.add_geometry(target_temp)

    # Set view parameters for better visualization
    vis.get_view_control().set_zoom(0.7)
    vis.poll_events()
    vis.update_renderer()

    # Capture and save image
    vis.capture_screen_image(filepath)
    vis.destroy_window()

    print(f"     Saved visualization to: {filepath}")

## Task 1: Global Registration Implementation

Global registration is used to roughly align two point clouds without requiring a good initial transformation. We'll use RANSAC-based registration with FPFH (Fast Point Feature Histograms) features.

### Approach:
1. **Downsample** point clouds using voxel grid filtering
2. **Estimate normals** for each point
3. **Compute FPFH features** to describe local geometry
4. **RANSAC registration** to find correspondences and estimate transformation

In [6]:
def global_registration(source, target, voxel_size=0.05, visualize=True):
    """
    Perform global registration using RANSAC with FPFH features.

    Args:
        source: Source point cloud
        target: Target point cloud
        voxel_size: Voxel size for downsampling (default: 0.05)
        visualize: Whether to visualize the result (default: True)

    Returns:
        transformation: 4x4 transformation matrix
        result: Registration result object with fitness and RMSE metrics
    """
    print(f"Global registration with voxel size: {voxel_size}")

    # Step 1: Downsample point clouds
    print("  1. Downsampling point clouds...")
    source_down = source.voxel_down_sample(voxel_size)
    target_down = target.voxel_down_sample(voxel_size)
    print(f"     Source: {len(source.points)} -> {len(source_down.points)} points")
    print(f"     Target: {len(target.points)} -> {len(target_down.points)} points")

    # Step 2: Estimate normals
    print("  2. Estimating normals...")
    source_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30))
    target_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30))

    # Step 3: Compute FPFH features
    print("  3. Computing FPFH features...")
    source_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        source_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100))
    target_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        target_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100))

    # Step 4: RANSAC registration
    print("  4. Running RANSAC registration...")
    distance_threshold = voxel_size * 1.5

    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down, target_down,
        source_fpfh, target_fpfh,
        mutual_filter=True,
        max_correspondence_distance=distance_threshold,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=4,
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
        ],
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500))

    print(f"\n  Results:")
    print(f"    Fitness: {result.fitness:.4f}")
    print(f"    Inlier RMSE: {result.inlier_rmse:.4f}")
    print(f"    Correspondence set size: {len(result.correspondence_set)}")

    if visualize:
        print("\n  Visualizing registration result...")
        draw_registrations(source, target, result.transformation, recolor=True)

    return result.transformation, result

### Test 1: Align r1.pcd and r2.pcd

In [7]:
# Load r1.pcd and r2.pcd
print("=" * 70)
print("TEST 1: Aligning r1.pcd and r2.pcd")
print("=" * 70)

source_r = o3d.io.read_point_cloud("global_registration/r1.pcd")
target_r = o3d.io.read_point_cloud("global_registration/r2.pcd")

print(f"\nLoaded point clouds:")
print(f"  r1.pcd: {len(source_r.points)} points")
print(f"  r2.pcd: {len(target_r.points)} points")

# Visualize before registration
print("\nSaving BEFORE registration visualization...")
draw_registrations(source_r, target_r, recolor=True, filename="r1_r2_before.png")

# Perform global registration
print("\n" + "-" * 70)
transformation_r, result_r = global_registration(source_r, target_r, voxel_size=0.05)

TEST 1: Aligning r1.pcd and r2.pcd

Loaded point clouds:
  r1.pcd: 198835 points
  r2.pcd: 137833 points

Saving BEFORE registration visualization...
     Saved visualization to: visualizations/r1_r2_before.png

----------------------------------------------------------------------
Global registration with voxel size: 0.05
  1. Downsampling point clouds...
     Source: 198835 -> 4760 points
     Target: 137833 -> 3440 points
  2. Estimating normals...
  3. Computing FPFH features...
  4. Running RANSAC registration...

  Results:
    Fitness: 0.6813
    Inlier RMSE: 0.0356
    Correspondence set size: 3243

  Visualizing registration result...
     Saved visualization to: visualizations/registration_000.png


### Test 2: Align car1.ply and car2.ply

In [8]:
# Load car1.ply and car2.ply
print("=" * 70)
print("TEST 2: Aligning car1.ply and car2.ply")
print("=" * 70)

source_car = o3d.io.read_point_cloud("global_registration/car1.ply")
target_car = o3d.io.read_point_cloud("global_registration/car2.ply")

print(f"\nLoaded point clouds:")
print(f"  car1.ply: {len(source_car.points)} points")
print(f"  car2.ply: {len(target_car.points)} points")

# Visualize before registration
print("\nSaving BEFORE registration visualization...")
draw_registrations(source_car, target_car, recolor=True, filename="car_before.png")

# Perform global registration with a smaller voxel size for better detail
print("\n" + "-" * 70)
transformation_car, result_car = global_registration(source_car, target_car, voxel_size=0.05)

TEST 2: Aligning car1.ply and car2.ply

Loaded point clouds:
  car1.ply: 166218 points
  car2.ply: 168373 points

Saving BEFORE registration visualization...
     Saved visualization to: visualizations/car_before.png

----------------------------------------------------------------------
Global registration with voxel size: 0.05
  1. Downsampling point clouds...
     Source: 166218 -> 3490 points
     Target: 168373 -> 3606 points
  2. Estimating normals...
  3. Computing FPFH features...
  4. Running RANSAC registration...

  Results:
    Fitness: 0.9977
    Inlier RMSE: 0.0182
    Correspondence set size: 3482

  Visualizing registration result...
     Saved visualization to: visualizations/registration_001.png


## Task 2 (Challenge): Local Registration with ICP

Local registration (ICP - Iterative Closest Point) refines the alignment from global registration. It requires a good initial transformation.

### Approach:
1. Use the transformation from global registration as initialization
2. Apply point-to-plane ICP for fine alignment
3. Compare results with and without ICP refinement

In [9]:
def local_registration_icp(source, target, initial_transformation, threshold=0.02, visualize=True):
    """
    Perform local registration using ICP to refine global registration.

    Args:
        source: Source point cloud
        target: Target point cloud
        initial_transformation: Initial 4x4 transformation matrix (from global registration)
        threshold: Distance threshold for ICP (default: 0.02)
        visualize: Whether to visualize the result (default: True)

    Returns:
        transformation: Refined 4x4 transformation matrix
        result: ICP registration result with fitness and RMSE metrics
    """
    print(f"\nLocal ICP registration with threshold: {threshold}")

    # Estimate normals for point-to-plane ICP
    print("  1. Estimating normals...")
    source.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    target.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))

    # Evaluate initial alignment
    print("  2. Evaluating initial alignment...")
    evaluation_init = o3d.pipelines.registration.evaluate_registration(
        source, target, threshold, initial_transformation)
    print(f"     Initial fitness: {evaluation_init.fitness:.4f}")
    print(f"     Initial RMSE: {evaluation_init.inlier_rmse:.4f}")

    # Run point-to-plane ICP
    print("  3. Running point-to-plane ICP...")
    result = o3d.pipelines.registration.registration_icp(
        source, target, threshold, initial_transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPlane())

    print(f"\n  ICP Results:")
    print(f"    Fitness: {result.fitness:.4f} (improvement: {result.fitness - evaluation_init.fitness:+.4f})")
    print(f"    Inlier RMSE: {result.inlier_rmse:.4f} (improvement: {result.inlier_rmse - evaluation_init.inlier_rmse:+.4f})")
    print(f"    Correspondence set size: {len(result.correspondence_set)}")

    if visualize:
        print("\n  Visualizing ICP registration result...")
        draw_registrations(source, target, result.transformation, recolor=True)

    return result.transformation, result

### Refine r1-r2 alignment with ICP

In [10]:
# Refine r1-r2 alignment with ICP
print("=" * 70)
print("REFINING: r1.pcd and r2.pcd with ICP")
print("=" * 70)

# Reload to avoid using point clouds with estimated normals from previous cells
source_r_icp = o3d.io.read_point_cloud("global_registration/r1.pcd")
target_r_icp = o3d.io.read_point_cloud("global_registration/r2.pcd")

transformation_r_icp, result_r_icp = local_registration_icp(
    source_r_icp, target_r_icp, transformation_r, threshold=0.02)

REFINING: r1.pcd and r2.pcd with ICP

Local ICP registration with threshold: 0.02
  1. Estimating normals...
  2. Evaluating initial alignment...
     Initial fitness: 0.3443
     Initial RMSE: 0.0115
  3. Running point-to-plane ICP...

  ICP Results:
    Fitness: 0.6211 (improvement: +0.2768)
    Inlier RMSE: 0.0066 (improvement: -0.0049)
    Correspondence set size: 123490

  Visualizing ICP registration result...
     Saved visualization to: visualizations/registration_002.png


### Refine car1-car2 alignment with ICP

In [11]:
# Refine car1-car2 alignment with ICP
print("=" * 70)
print("REFINING: car1.ply and car2.ply with ICP")
print("=" * 70)

# Reload to avoid using point clouds with estimated normals from previous cells
source_car_icp = o3d.io.read_point_cloud("global_registration/car1.ply")
target_car_icp = o3d.io.read_point_cloud("global_registration/car2.ply")

transformation_car_icp, result_car_icp = local_registration_icp(
    source_car_icp, target_car_icp, transformation_car, threshold=0.02)

REFINING: car1.ply and car2.ply with ICP

Local ICP registration with threshold: 0.02
  1. Estimating normals...
  2. Evaluating initial alignment...
     Initial fitness: 0.9786
     Initial RMSE: 0.0081
  3. Running point-to-plane ICP...

  ICP Results:
    Fitness: 0.9876 (improvement: +0.0091)
    Inlier RMSE: 0.0054 (improvement: -0.0027)
    Correspondence set size: 164164

  Visualizing ICP registration result...
     Saved visualization to: visualizations/registration_003.png


### Summary and Comparison

The complete pipeline consists of:
1. **Global Registration (RANSAC + FPFH)**: Roughly aligns point clouds without initial pose
2. **Local Registration (ICP)**: Refines alignment using iterative closest point matching

**Key Benefits of ICP Refinement:**
- Lower RMSE (better alignment accuracy)
- Higher fitness score (more point correspondences)
- Corrects small misalignments from global registration

**When to use each method:**
- **Global only**: When point clouds are very different or you only need rough alignment
- **Global + ICP**: For precise alignment and better reconstruction quality